# Chương 1 — EDA chuyên sâu dữ liệu bảng: Adult Census Income

## Mục tiêu

Chương này kiểm tra điều kiện dữ liệu, phân phối từng biến số, tương quan Pearson/Spearman, cấu trúc missing và giả thuyết MCAR. Mọi kết luận phải dựa trên kết quả notebook chạy thực tế, không điền số liệu giả định.

**Nguồn dữ liệu:** [Adult Census Income trên Hugging Face](https://huggingface.co/datasets/scikit-learn/adult-census-income). Bộ dữ liệu có 32.561 quan sát, 14 thuộc tính đầu vào và nhãn thu nhập. Dấu `?` trong ba thuộc tính phân loại được chuẩn hóa thành giá trị thiếu `NaN`.

> Lưu ý: dữ liệu mô tả hồ sơ điều tra dân số Hoa Kỳ năm 1994; không được diễn giải kết quả như quan hệ nhân quả hoặc áp dụng trực tiếp cho bối cảnh hiện tại.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p for p in _candidates if (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Không tìm thấy thư mục src/. Hãy mở notebook từ thư mục repository gốc.")
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data_loading import load_adult_dataset
from src.tabular_analysis import split_feature_types

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
print("Repository:", REPO_ROOT)

## 1. Tải dữ liệu và kiểm tra điều kiện đề bài

Bộ dữ liệu được tải tự động từ Hugging Face ở lần chạy đầu tiên và lưu cache bên ngoài mã nguồn. Không đưa dữ liệu thô vào GitHub.

In [ ]:
df = load_adult_dataset()
X, numeric_cols, categorical_cols = split_feature_types(df, target="income")

n_rows, n_features = len(df), X.shape[1]
missing_rate = X.isna().mean().sort_values(ascending=False)
validation = {
    "records >= 10,000": n_rows >= 10_000,
    "predictor attributes >= 10": n_features >= 10,
    "both numerical and categorical": len(numeric_cols) > 0 and len(categorical_cols) > 0,
    "at least one missing rate >= 5%": bool((missing_rate >= 0.05).any()),
}
print(f"Shape: {df.shape}; predictors: {n_features}; numeric: {len(numeric_cols)}; categorical: {len(categorical_cols)}")
display(pd.DataFrame({"requirement": validation.keys(), "passed": validation.values()}))
display(missing_rate.rename("missing_rate").to_frame().style.format({"missing_rate": "{:.2%}"}))
assert all(validation.values()), "Dataset hiện không đạt các ngưỡng bắt buộc của đề bài."

## 2. Tổng quan cấu trúc và thống kê mô tả

- `shape`, kiểu dữ liệu và nhãn cho biết quy mô, cấu trúc tổng quát.
- `describe` tóm tắt trung tâm, độ phân tán và các phân vị của biến số.
- Kiểm tra nhãn trước khi mô hình hóa; không dùng biến mục tiêu làm đầu vào.

In [ ]:
display(df.head())
display(df.dtypes.rename("dtype").to_frame())
display(df[numeric_cols].describe().T)
print("Target distribution:")
display(df["income"].value_counts(dropna=False).rename("count").to_frame())

## 3. Kiểm định phân phối từng thuộc tính số

Với mỗi biến có `n > 5.000`, sử dụng kiểm định D’Agostino–Pearson (`scipy.stats.normaltest`).

- $H_0$: dữ liệu phù hợp với phân phối chuẩn.
- $H_1$: dữ liệu không phù hợp với phân phối chuẩn.
- Quy ước $\alpha=0.05$: `p < 0.05` là bằng chứng chống lại giả thuyết chuẩn.

Với cỡ mẫu lớn, kiểm định rất nhạy; cần kết hợp histogram/KDE và không chỉ dựa vào p-value.

$$K^2=Z_1^2+Z_2^2\ \sim\ \chi^2_2,\qquad b_1=\frac{m_3}{m_2^{3/2}},\quad b_2=\frac{m_4}{m_2^{2}},\quad m_k=\frac1n\sum_{i=1}^n(x_i-\bar x)^k$$

$Z_1$ và $Z_2$ là các biến đổi chuẩn hóa của độ lệch $b_1$ và độ nhọn $b_2$ (D’Agostino–Pearson). Shapiro–Wilk được dùng khi $n\le 5000$.

In [ ]:
from scipy.stats import normaltest

normality_rows = []
for col in numeric_cols:
    values = df[col].dropna().astype(float)
    if len(values) > 5_000:
        statistic, p_value = normaltest(values)
        test_name = "D'Agostino-Pearson"
    else:
        from scipy.stats import shapiro
        statistic, p_value = shapiro(values.sample(min(len(values), 5_000), random_state=RANDOM_SEED))
        test_name = "Shapiro-Wilk"
    normality_rows.append({
        "feature": col, "n": len(values), "test": test_name,
        "statistic": float(statistic), "p_value": float(p_value),
        "decision": "Không chuẩn (bác bỏ H0)" if p_value < 0.05 else "Chưa đủ bằng chứng bác bỏ chuẩn"
    })
normality_results = pd.DataFrame(normality_rows).sort_values("p_value")
display(normality_results)

In [ ]:
fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(10, 3.2 * len(numeric_cols)))
if len(numeric_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col].dropna(), bins=40, kde=True, stat="density", ax=ax)
    ax.set_title(f"Phân phối: {col}")
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

**Phân tích kết quả:** đối chiếu bảng kiểm định với hình phân phối. Các biến vốn lệch mạnh (ví dụ biến có nhiều giá trị 0 và đuôi phải dài) thường không phù hợp giả định chuẩn. Đây là một căn cứ để cân nhắc Robust Scaling hoặc Quantile Transform ở chương tiếp theo; không được tự động kết luận rằng một scaler “tốt nhất” chỉ từ kiểm định chuẩn.

## 4. Phân tích tương quan Pearson và Spearman

- Pearson đo tương quan tuyến tính giữa hai biến số.
- Spearman đo tương quan đơn điệu dựa trên thứ hạng, ít nhạy hơn với ngoại lệ và quan hệ phi tuyến đơn điệu.
- Cặp có $|r| > 0.9$ được đánh dấu để xem xét đa cộng tuyến; tương quan cao không tự động chứng minh một biến phải bị xóa.

$$r=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sqrt{\sum_i(x_i-\bar x)^2\,\sum_i(y_i-\bar y)^2}},\qquad \rho_s=r\bigl(\operatorname{rank}(x),\operatorname{rank}(y)\bigr)$$

$r$ là hệ số Pearson; $\rho_s$ là hệ số Spearman, tức Pearson áp dụng trên thứ hạng.

In [ ]:
pearson_corr = df[numeric_cols].corr(method="pearson")
spearman_corr = df[numeric_cols].corr(method="spearman")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Pearson correlation")
sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Spearman correlation")
plt.tight_layout(); plt.show()

strong_pairs = []
for i, left in enumerate(numeric_cols):
    for right in numeric_cols[i + 1:]:
        value = pearson_corr.loc[left, right]
        if abs(value) > 0.9:
            strong_pairs.append({"feature_1": left, "feature_2": right, "pearson_r": value})
strong_correlations = pd.DataFrame(strong_pairs, columns=["feature_1", "feature_2", "pearson_r"])
print("Các cặp có |Pearson r| > 0.9:")
display(strong_correlations if not strong_correlations.empty else pd.DataFrame({"note": ["Không phát hiện cặp vượt ngưỡng 0.9."]}))

## 5. Phân tích dữ liệu thiếu bằng missingno

Ma trận missingno cho thấy vị trí các ô thiếu theo từng quan sát. Bảng tỷ lệ missing dùng để xác định cột ảnh hưởng lớn nhất. Giá trị `?` đã được chuyển về `NaN` trong hàm tải dữ liệu.

In [ ]:
missing_table = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_rate": X.isna().mean(),
}).sort_values("missing_rate", ascending=False)
display(missing_table.style.format({"missing_rate": "{:.2%}"}))

import missingno as msno
fig = msno.matrix(X, figsize=(12, 5), sparkline=False)
plt.title("Missing data matrix — Adult Census Income")
plt.show()

## 6. Little’s MCAR test

Kiểm định Little có giả thuyết không $H_0$: dữ liệu thiếu hoàn toàn ngẫu nhiên (MCAR). Nếu `p < 0.05`, bác bỏ MCAR. Do dữ liệu chứa biến phân loại, notebook ordinal-code các biến phân loại để có ma trận số đầu vào cho `pyampute`; đây là xấp xỉ cần nêu rõ, vì kiểm định Little dựa trên giả định phân phối đa biến và mã hóa thứ bậc không bảo toàn khoảng cách giữa category.

$$d^2=\sum_{j=1}^{J}n_j\left(\bar{\mathbf x}_j^{\,obs}-\hat{\boldsymbol\mu}^{\,obs}_j\right)^{\!\top}\hat\Sigma_j^{-1}\left(\bar{\mathbf x}_j^{\,obs}-\hat{\boldsymbol\mu}^{\,obs}_j\right)\ \sim\ \chi^2_{\,\sum_j k_j-p}$$

$J$ là số mẫu hình thiếu dữ liệu, $n_j$ là số dòng của mẫu hình $j$, $k_j$ là số biến quan sát được trong mẫu hình đó, $p$ là tổng số biến. $\hat{\boldsymbol\mu}$ và $\hat\Sigma$ được ước lượng bằng EM.

In [ ]:
from src.tabular_analysis import little_mcar_test

# A reproducible sample keeps the statistical test responsive on a laptop.
mcar_input = df.sample(min(len(df), 8_000), random_state=RANDOM_SEED).copy()
little_mcar_p_value = little_mcar_test(mcar_input, target="income")
print(f"Little's MCAR test p-value = {little_mcar_p_value:.6g}")
if little_mcar_p_value < 0.05:
    print("Kết luận ở alpha=0.05: bác bỏ H0; có bằng chứng dữ liệu không MCAR.")
else:
    print("Kết luận ở alpha=0.05: chưa đủ bằng chứng bác bỏ H0; không đồng nghĩa đã chứng minh MCAR.")

## 7. Phân biệt MCAR, MAR và MNAR — bằng chứng và giới hạn

- **MCAR:** xác suất thiếu không phụ thuộc cả dữ liệu đã quan sát lẫn chưa quan sát.
- **MAR:** sau khi điều kiện hóa theo biến đã quan sát, xác suất thiếu không còn phụ thuộc giá trị đang thiếu.
- **MNAR:** xác suất thiếu còn phụ thuộc chính giá trị chưa quan sát.

Little’s test chỉ kiểm tra giả thuyết MCAR; nó không tự phân biệt MAR với MNAR. Bước tiếp theo kiểm tra xem trạng thái thiếu ở các cột chính có liên hệ với các thuộc tính quan sát được hay không. Liên hệ có ý nghĩa là bằng chứng ủng hộ khả năng MAR, nhưng không chứng minh MAR. MNAR cần thông tin ngoài dữ liệu hoặc giả định chuyên môn.

In [ ]:
from scipy.stats import chi2_contingency

missingness_associations = []
for missing_col in [c for c in X.columns if X[c].isna().any()]:
    indicator = X[missing_col].isna().astype(int)
    for observed_col in X.columns:
        if observed_col == missing_col:
            continue
        observed = X[observed_col]
        if pd.api.types.is_numeric_dtype(observed):
            # Bin numerical covariates into quantiles for a chi-square association check.
            grouped = pd.qcut(observed, q=5, duplicates="drop")
        else:
            grouped = observed.astype("string").fillna("__MISSING__")
            top = grouped.value_counts().head(12).index
            grouped = grouped.where(grouped.isin(top), "__OTHER__")
        table = pd.crosstab(indicator, grouped)
        if table.shape[0] == 2 and table.shape[1] >= 2:
            chi2_stat, p_value, dof, _ = chi2_contingency(table)
            missingness_associations.append({
                "missing_column": missing_col,
                "observed_covariate": observed_col,
                "chi2": float(chi2_stat), "dof": int(dof), "p_value": float(p_value)
            })
association_results = pd.DataFrame(missingness_associations).sort_values("p_value")
display(association_results.head(20))

### Kết luận EDA cần hoàn thiện sau khi chạy

1. Chỉ ra các cột có tỷ lệ missing đạt ngưỡng 5% trở lên.
2. Ghi p-value của từng kiểm định phân phối và nêu biến nào lệch chuẩn rõ nhất.
3. Nêu cặp Pearson/Spearman có tương quan mạnh, nếu có.
4. Ghi p-value Little’s MCAR và kết luận đúng theo giả thuyết kiểm định.
5. Dựa trên bảng liên hệ trạng thái thiếu với biến quan sát để thảo luận MAR; ghi rõ MNAR không thể kết luận chỉ từ dữ liệu quan sát.

Không sao chép các giá trị p-value trước khi notebook chạy trên dữ liệu thực.